# XLKD-03: Dense Mapped Logit-KD Training Readiness

XLKD-02 đã chứng minh loss, backward, gradient ratio, fixed-batch overfit và
GPU memory đều đạt gate. XLKD-03 khóa production engine trước full run:

- Teacher ExpansionNet v2 frozen;
- Student S1 khởi tạo từ baseline-best seed 42;
- Dense On-the-Fly Cross-Vocabulary Mapped Logit KD;
- `T=4`, `0,5 CE_human + 0,5 KL×T²`;
- physical batch 1, accumulation 64, effective batch 64;
- AdamW, warmup-cosine, FP16 Student và FP32 Teacher;
- atomic optimizer-step resume;
- discarded pilot, không giữ trọng số;
- Validation không mở trong readiness; Test luôn `SEALED`.

Notebook cũng khóa lộ trình từ XLKD-03 đến Multi-Level KD, đánh giá cuối,
triển khai mobile và hoàn thiện báo cáo.

## Kaggle inputs

1. XLKD-00 output.
2. XLKD-02 output.
3. Phase 01 Formal Output.
4. Phase 02A Shared Student Pipeline.
5. Phase 02B S1 seed-42 baseline output.
6. Phase 03B ExpansionNet v2 preflight output.
7. Phase 03E ExpansionNet v2 handoff output.
8. ExpansionNet v2 offline model package.
9. MS COCO 2014 image dataset.

Runtime: GPU T4 hoặc mạnh hơn, Internet Off.


## Cell 1 — Authorization, manifests và immutable inputs


In [1]:
import copy, csv, gc, hashlib, json, math, os, pickle, platform, random, sys, time
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
DATASET_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//xlkd03_dense_mapped_training_readiness",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
LOG = OUT / "logs"
RESULT = OUT / "results"
CHECKPOINT = OUT / "checkpoints"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, LOG, RESULT, CHECKPOINT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

_DISCOVERY_INDEX = {}
for _candidate in NOTEBOOK_ROOT.rglob("*"):
    if _candidate.is_file():
        _DISCOVERY_INDEX.setdefault(_candidate.name, []).append(
            _candidate.resolve()
        )

def exactly_one(name):
    matches = sorted(_DISCOVERY_INDEX.get(name, []))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]

def verify_csv_manifest(root, reference):
    path = resolve_inside(root, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if "rows" in reference:
        assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

def find_expansionnet_manifest():
    preferred = INPUT_ROOT / "models"
    search_root = preferred if preferred.is_dir() else INPUT_ROOT
    matches = []
    for candidate in search_root.rglob("MODEL_SOURCE_MANIFEST.json"):
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get("model_id") == "expansionnet_v2":
            matches.append(candidate.resolve())
    assert len(matches) == 1, matches
    return matches[0]

if not torch.cuda.is_available():
    raise RuntimeError(
        "XLKD-03 yêu cầu GPU T4 hoặc mạnh hơn. Hãy bật Accelerator rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

X02_PATH = exactly_one("XLKD02_DENSE_MAPPED_LOSS_PREFLIGHT_COMPLETE.json")
AUTH_PATH = exactly_one(
    "XLKD03_DENSE_MAPPED_LOGIT_KD_TRAINING_READINESS_AUTHORIZED.json"
)
X00_AUTH_PATH = exactly_one(
    "XLKD01_GPU_PROBABILITY_MASS_PREFLIGHT_AUTHORIZED.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2A_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
S1_PATH = exactly_one("S1_SEED42_FULL_RUN_COMPLETE.json")
P3B_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
P3E_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
MODEL_MANIFEST_PATH = find_expansionnet_manifest()
MODEL_ROOT = MODEL_MANIFEST_PATH.parent

X02, AUTH, X00_AUTH, P1, P2A, S1, P3B, P3E = map(
    read_json,
    [X02_PATH, AUTH_PATH, X00_AUTH_PATH, P1_PATH, P2A_PATH,
     S1_PATH, P3B_PATH, P3E_PATH],
)
assert X02["status"] == "complete" and X02["overall_gate"] is True
assert X02["decision"] == "authorize_xlkd03_training_readiness"
assert X02["full_training_authorized"] is False
assert AUTH["status"] == "authorized"
assert AUTH["required_next"].startswith("XLKD-03")
assert AUTH["full_training_authorized"] is False
assert float(AUTH["temperature"]) == 4.0
assert float(AUTH["alpha_human_ce"]) == 0.5
assert float(AUTH["alpha_kd"]) == 0.5
assert int(AUTH["physical_batch_size"]) == 1
assert X00_AUTH["status"] == "authorized"
assert P1["status"] == "ready"
assert P2A["status"] == "ready"
assert S1["status"] == "complete"
assert S1["student_id"] == "S1" and int(S1["seed"]) == 42
assert P3B["status"] == "ready" and P3B["teacher"] == "ExpansionNet v2"
assert P3E["status"] == "complete"
for marker in [X02, AUTH, X00_AUTH, S1, P3B, P3E]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2A["data_policy"]["test_set_status"] == "sealed"
assert P2A["data_policy"]["test_set_used"] is False

x02_rows = verify_csv_manifest(X02_PATH.parent, X02["manifest"]["csv"])
assert len(x02_rows) == 11
p2a_rows = verify_csv_manifest(P2A_PATH.parent, P2A["manifests"]["csv"])
assert len(p2a_rows) == 38

decision = read_json(X02_PATH.parent / "results" / "xlkd02_decision.json")
assert decision["overall_gate"] is True
assert all(decision["gates"].values())
LOSS_CONTRACT_PATH = (
    X02_PATH.parent / "configs" / "xlkd02_executed_loss_contract.json"
).resolve()
LOSS_CONTRACT = read_json(LOSS_CONTRACT_PATH)
assert LOSS_CONTRACT["status"] == "locked"
assert LOSS_CONTRACT["temperature"] == 4.0
assert LOSS_CONTRACT["full_training_authorized"] is False

mapping_ref = X00_AUTH["mapping_contract"]
MAPPING_PATH = resolve_inside(X00_AUTH_PATH.parent, mapping_ref["relative_path"])
assert sha256(MAPPING_PATH) == mapping_ref["sha256"]
MAPPING_PAYLOAD = read_json(MAPPING_PATH)
mapping = {
    int(source): int(target)
    for source, target in MAPPING_PAYLOAD["mapping"].items()
}
assert len(mapping) == 9959
assert MAPPING_PAYLOAD["mapping_collisions"] == 0

model_manifest = read_json(MODEL_MANIFEST_PATH)
assert model_manifest["model_id"] == "expansionnet_v2"

def verify_model_record(record):
    target = resolve_inside(MODEL_ROOT, record["relative_path"])
    assert target.is_file()
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256(target) == record["sha256"]
    return target

CHECKPOINT_PATH = verify_model_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_model_record(model_manifest["vocabulary"])
for record in model_manifest["required_files"]:
    verify_model_record(record)
for record in model_manifest["integration_patches"]:
    target = resolve_inside(MODEL_ROOT, record["file"])
    assert target.is_file() and sha256(target) == record["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_tokens = pickle.load(handle)
NATIVE_WORD2IDX = native_tokens["word2idx_dict"]
NATIVE_IDX2WORD = native_tokens["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX[native_tokens["sos_str"]])
NATIVE_EOS = int(NATIVE_WORD2IDX[native_tokens["eos_str"]])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
NATIVE_UNK = int(NATIVE_WORD2IDX["UNK"])
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (78, 79, 77, 80)

PSEUDO_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
).resolve()
PSEUDO_SUMMARY_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
).resolve()
PSEUDO_SUMMARY = read_json(PSEUDO_SUMMARY_PATH)
assert PSEUDO_SUMMARY["records"] == 113287
assert sha256(PSEUDO_PATH) == PSEUDO_SUMMARY["jsonl_sha256"]
assert PSEUDO_SUMMARY["validation_overlap"] == 0
assert PSEUDO_SUMMARY["held_out_test_overlap"] == 0

BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")

write_json(AUDIT / "xlkd03_cell01_authorization.json", {
    "schema_version": "xlkd03_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd02_manifest_rows": len(x02_rows),
    "phase2a_manifest_rows": len(p2a_rows),
    "xlkd02_gates": decision["gates"],
    "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "student_checkpoint_sha256": sha256(BASELINE_CHECKPOINT),
    "mapping_tokens": len(mapping),
    "pseudo_caption_rows": 113287,
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "strict_deterministic": True,
    "checkpoint_payload_read": False,
    "dataset_read": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 1 — AUTHORIZATION AND INTEGRITY")
print("=" * 78)
print("Status                    : PASSED")
print("XLKD-02 manifest          : 11 / 11 VERIFIED")
print("Phase 02A manifest        : 38 / 38 VERIFIED")
print("XLKD-02 gates             : 9 / 9 PASSED")
print("Vocabulary mapping        : 9,959 VERIFIED")
print("Pseudo-caption corpus     : 113,287 VERIFIED")
print("Teacher / Student ckpt    : VERIFIED / VERIFIED")
print("Validation / Test         : NOT READ / SEALED")
print("Full training             : NOT STARTED")
print("Progress                  : 1 / 7 code cells")


XLKD-03 CELL 1 — AUTHORIZATION AND INTEGRITY
Status                    : PASSED
XLKD-02 manifest          : 11 / 11 VERIFIED
Phase 02A manifest        : 38 / 38 VERIFIED
XLKD-02 gates             : 9 / 9 PASSED
Vocabulary mapping        : 9,959 VERIFIED
Pseudo-caption corpus     : 113,287 VERIFIED
Teacher / Student ckpt    : VERIFIED / VERIFIED
Validation / Test         : NOT READ / SEALED
Full training             : NOT STARTED
Progress                  : 1 / 7 code cells


## Cell 2 — Full Train+Restval pipeline và exact-resume sampler


In [2]:
STARTED = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision.transforms import InterpolationMode
from torchvision.transforms import RandomResizedCrop
from torchvision.transforms import functional as TF

TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")

def find_image_leaf(name, prefix):
    matches = []
    for path in DATASET_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "train")],
)
TRAIN_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    TRAIN_PATHS[image_id] = (leaf / Path(row["image_path"]).name).resolve()
assert len(TRAIN_PATHS) == 113287

TRAIN_CAPTIONS = defaultdict(list)
for arrow_batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in arrow_batch.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append((
            int(row["annotation_id"]),
            tuple(int(value) for value in row["token_ids"]),
        ))
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()
assert len(TRAIN_CAPTIONS) == 113287

PSEUDO_BY_IMAGE = {}
with PSEUDO_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        image_id = int(row["image_id"])
        teacher_ids = tuple(int(value) for value in row["teacher_token_ids"])
        student_ids = tuple(int(value) for value in row["student_token_ids"])
        assert len(teacher_ids) == len(student_ids)
        assert [mapping[value] for value in teacher_ids] == list(student_ids)
        assert image_id not in PSEUDO_BY_IMAGE
        PSEUDO_BY_IMAGE[image_id] = (teacher_ids, student_ids)
assert set(PSEUDO_BY_IMAGE) == set(TRAIN_PATHS)
TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|xlkd03"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8], "big"
    ) % (2 ** 63 - 1)

def dual_train_transform(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = RandomResizedCrop.get_params(
            image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
        )
        do_flip = bool(torch.rand(()) < 0.5)
    teacher = TF.resized_crop(
        image, top, left, height, width, [384, 384],
        InterpolationMode.BILINEAR, antialias=True,
    )
    student = TF.resized_crop(
        image, top, left, height, width, [224, 224],
        InterpolationMode.BICUBIC, antialias=True,
    )
    if do_flip:
        teacher, student = TF.hflip(teacher), TF.hflip(student)
    teacher = TF.normalize(TF.to_tensor(teacher), IMAGENET_MEAN, IMAGENET_STD)
    student = TF.normalize(TF.to_tensor(student), IMAGENET_MEAN, IMAGENET_STD)
    return teacher, student

class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, human_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        teacher_ids, student_ids = PSEUDO_BY_IMAGE[image_id]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            teacher_image, student_image = dual_train_transform(
                source.convert("RGB"), self.epoch, image_id
            )
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "teacher_image": teacher_image,
            "student_image": student_image,
            "teacher_ids": torch.tensor(teacher_ids, dtype=torch.long),
            "student_pseudo_ids": torch.tensor(student_ids, dtype=torch.long),
            "human_ids": torch.tensor(human_ids, dtype=torch.long),
        }

class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(self.length, generator=generator).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor

def production_collate(rows):
    assert len(rows) == 1
    row = rows[0]
    return {
        "image_ids": torch.tensor([row["image_id"]], dtype=torch.long),
        "annotation_ids": torch.tensor([row["annotation_id"]], dtype=torch.long),
        "teacher_image": row["teacher_image"].unsqueeze(0),
        "student_image": row["student_image"].unsqueeze(0),
        "teacher_ids": row["teacher_ids"].unsqueeze(0),
        "student_pseudo_ids": row["student_pseudo_ids"].unsqueeze(0),
        "human_ids": row["human_ids"].unsqueeze(0),
    }

TRAIN_DATASET = ProductionTrainDataset()

def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    return DataLoader(
        TRAIN_DATASET,
        batch_size=1,
        sampler=EpochCursorSampler(len(TRAIN_DATASET), epoch, cursor),
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=production_collate,
    )

batch_a = next(iter(build_train_loader(0)))
batch_b = next(iter(build_train_loader(0)))
for key in [
    "image_ids", "annotation_ids", "teacher_image", "student_image",
    "teacher_ids", "student_pseudo_ids", "human_ids",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert batch_a["teacher_image"].shape == (1, 3, 384, 384)
assert batch_a["student_image"].shape == (1, 3, 224, 224)
assert torch.isfinite(batch_a["teacher_image"]).all()
assert torch.isfinite(batch_a["student_image"]).all()

write_json(AUDIT / "xlkd03_cell02_data_pipeline.json", {
    "schema_version": "xlkd03_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "train_images_per_epoch": len(TRAIN_DATASET),
    "human_caption_sampling": "one per image per epoch, cyclic",
    "teacher_sequence_per_image": 1,
    "physical_batch_size": 1,
    "shared_stateless_geometry": True,
    "teacher_resolution": [384, 384],
    "student_resolution": [224, 224],
    "teacher_student_sequence_alignment": "113287 / 113287",
    "two_loader_first_batch_exact": True,
    "resume_cursor_supported": True,
    "validation_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE")
print("=" * 78)
print("Status                    : PASSED")
print("Train images / epoch      : 113,287")
print("Human caption / image     : 1 CYCLIC")
print("Teacher sequence / image  : 1 FIXED")
print("Sequence alignment        : 113,287 / 113,287")
print("Physical batch            : 1")
print("Teacher / Student input   : 384 / 224")
print("Stateless shared geometry : True")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 7 code cells")


XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE
Status                    : PASSED
Train images / epoch      : 113,287
Human caption / image     : 1 CYCLIC
Teacher sequence / image  : 1 FIXED
Sequence alignment        : 113,287 / 113,287
Physical batch            : 1
Teacher / Student input   : 384 / 224
Stateless shared geometry : True
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 7 code cells


## Cell 3 — Frozen Teacher, S1 baseline-best và production optimizer


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048, num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
teacher_payload = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
incompatible = TEACHER.load_state_dict(
    teacher_payload["model_state_dict"], strict=True
)
assert not incompatible.missing_keys and not incompatible.unexpected_keys
del teacher_payload
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
assert all(not parameter.requires_grad for parameter in TEACHER.parameters())

DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )

STUDENT_ENCODER = S1Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
baseline_payload = torch.load(
    BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
)
assert baseline_payload["schema_version"] == (
    "phase2_s1_full_training_checkpoint_v1"
)
STUDENT_ENCODER.load_state_dict(
    baseline_payload["encoder_state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    baseline_payload["decoder_state_dict"], strict=True
)
BASE_ENCODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
BASE_DECODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
del baseline_payload
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3231024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328
assert sum(p.numel() for p in TEACHER.parameters()) > 100_000_000

mapped_src = torch.tensor(
    sorted(mapping), dtype=torch.long, device=device
)
mapped_dst = torch.tensor(
    [mapping[index] for index in sorted(mapping)],
    dtype=torch.long, device=device,
)
assert mapped_src.numel() == mapped_dst.numel() == 9959

def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 46

def reset_student_baseline():
    STUDENT_ENCODER.load_state_dict(BASE_ENCODER_STATE, strict=True)
    STUDENT_DECODER.load_state_dict(BASE_DECODER_STATE, strict=True)
    for parameter in list(STUDENT_ENCODER.parameters()) + list(STUDENT_DECODER.parameters()):
        parameter.grad = None

def teacher_mapped_distribution(batch, temperature=4.0):
    images = batch["teacher_image"].to(device, non_blocking=True)
    full = batch["teacher_ids"].to(device, non_blocking=True)
    decoder_input = full[:, :-1]
    labels = full[:, 1:]
    num_pads = (decoder_input == NATIVE_PAD).sum(dim=1).tolist()
    # Do not use inference_mode here: q is consumed by F.kl_div while
    # autograd records the Student branch. A tensor created in
    # inference_mode cannot be saved for that backward pass. no_grad keeps
    # the Teacher frozen but returns ordinary tensors that are valid targets.
    with torch.no_grad():
        logits = TEACHER(
            enc_x=images,
            dec_x=decoder_input,
            enc_x_num_pads=[0] * images.shape[0],
            dec_x_num_pads=[int(value) for value in num_pads],
            apply_log_softmax=False,
            mode="forward",
        )
        assert logits.shape == (
            images.shape[0], decoder_input.shape[1], 10000
        )
        probabilities = torch.softmax(logits.float() / temperature, dim=-1)
        selected = probabilities.index_select(-1, mapped_src)
        dense = torch.zeros(
            *selected.shape[:-1], 10080,
            dtype=selected.dtype, device=device,
        )
        index = mapped_dst.view(1, 1, -1).expand_as(selected)
        # XLKD-00 proved that the map is collision-free, so scatter (not
        # scatter_add) is mathematically equivalent and deterministic.
        dense.scatter_(-1, index, selected)
        mass = dense.sum(dim=-1)
        q = dense / mass.unsqueeze(-1)
        mask = labels.ne(NATIVE_PAD)
    return q.detach(), mask.detach(), mass.detach()


STARTED_ENGINE = time.perf_counter()

TEMPERATURE = 4.0
ALPHA_HUMAN = 0.5
ALPHA_KD = 0.5
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 64
EFFECTIVE_BATCH = 64
STEPS_PER_EPOCH = math.ceil(113287 / EFFECTIVE_BATCH)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())
all_student_parameters = (
    backbone_parameters + adapter_parameters + decoder_parameters
)

def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4,
    )
    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / max(1, WARMUP_STEPS)
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler

def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }

def optimizer_step(microbatches, optimizer, scheduler, scaler, retry_limit=10):
    human_total = sum(
        int(batch["human_ids"][:, 1:].ne(0).sum())
        for batch in microbatches
    )
    kd_total = sum(
        int(batch["student_pseudo_ids"][:, 1:].ne(0).sum())
        for batch in microbatches
    )
    assert human_total > 0 and kd_total > 0
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        human_sum_value = 0.0
        kd_sum_value = 0.0
        mapped_values = []
        for cpu_batch in microbatches:
            q_teacher, kd_mask, mapped_mass = (
                teacher_mapped_distribution(
                    cpu_batch, temperature=TEMPERATURE
                )
            )
            current = to_device(cpu_batch)
            human = current["human_ids"]
            pseudo = current["student_pseudo_ids"]
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = STUDENT_ENCODER(current["student_image"])
                human_logits = STUDENT_DECODER(
                    human[:, :-1], memory, human[:, :-1].ne(0)
                )
                pseudo_logits = STUDENT_DECODER(
                    pseudo[:, :-1], memory, pseudo[:, :-1].ne(0)
                )
                human_sum = F.cross_entropy(
                    human_logits.float().reshape(-1, 10080),
                    human[:, 1:].reshape(-1),
                    ignore_index=0, reduction="sum",
                )
                student_log_prob = F.log_softmax(
                    pseudo_logits.float() / TEMPERATURE, dim=-1
                )
                assert torch.equal(kd_mask, pseudo[:, 1:].ne(0))
                kd_sum = (
                    F.kl_div(
                        student_log_prob, q_teacher, reduction="none"
                    ).sum(dim=-1)[kd_mask].sum()
                    * (TEMPERATURE ** 2)
                )
                normalized = (
                    ALPHA_HUMAN * human_sum / human_total
                    + ALPHA_KD * kd_sum / kd_total
                )
            scaler.scale(normalized / BACKWARD_DIVISOR).backward()
            human_sum_value += float(human_sum.detach())
            kd_sum_value += float(kd_sum.detach())
            mapped_values.extend(
                mapped_mass[kd_mask].detach().cpu().double().tolist()
            )
            del (
                current, memory, human_logits, pseudo_logits,
                q_teacher, kd_mask, mapped_mass, student_log_prob,
                human_sum, kd_sum, normalized,
            )
        scaler.unscale_(optimizer)
        for parameter in all_student_parameters:
            if parameter.grad is not None:
                parameter.grad.mul_(BACKWARD_DIVISOR)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            all_student_parameters, 1.0
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        scheduler.step()
        human_ce = human_sum_value / human_total
        kd_kl_t2 = kd_sum_value / kd_total
        return {
            "combined_loss": (
                ALPHA_HUMAN * human_ce + ALPHA_KD * kd_kl_t2
            ),
            "human_ce": human_ce,
            "mapped_kl_t2": kd_kl_t2,
            "human_tokens": human_total,
            "kd_positions": kd_total,
            "mapped_mass_mean": float(np.mean(mapped_values)),
            "mapped_mass_minimum": float(np.min(mapped_values)),
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }

optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771
assert len(optimizer.param_groups) == 3

write_json(CONFIG / "xlkd03_optimizer_contract.json", {
    "schema_version": "xlkd03_optimizer_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "optimizer": "AdamW",
    "learning_rates": {
        "backbone": 1e-4,
        "adapter": 3e-4,
        "decoder": 3e-4,
    },
    "weight_decay": 1e-4,
    "physical_batch": 1,
    "accumulation": ACCUMULATION,
    "effective_batch": EFFECTIVE_BATCH,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "maximum_epochs": MAX_EPOCHS,
    "total_steps": TOTAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "minimum_lr_factor": 0.05,
    "mixed_precision": "Student FP16 autocast plus GradScaler; Teacher FP32 no-grad",
    "gradient_clip": 1.0,
    "overflow_retry_limit": 10,
    "backward_divisor": BACKWARD_DIVISOR,
    "temperature": TEMPERATURE,
    "alpha_human_ce": ALPHA_HUMAN,
    "alpha_kd": ALPHA_KD,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "xlkd03_cell03_model_optimizer.json", {
    "schema_version": "xlkd03_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "teacher_strict_load": True,
    "teacher_frozen": True,
    "student_strict_load": True,
    "student_initialization": "S1 seed-42 baseline best",
    "student_parameters": 8990352,
    "optimizer_parameter_groups": 3,
    "physical_batch": 1,
    "accumulation": ACCUMULATION,
    "effective_batch": EFFECTIVE_BATCH,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "total_steps": TOTAL_STEPS,
    "scheduler_advanced": False,
    "optimizer_step_performed": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED_ENGINE,
})
print("=" * 78)
print("XLKD-03 CELL 3 — MODEL AND OPTIMIZER ENGINE")
print("=" * 78)
print("Status                    : PASSED")
print("Teacher                   : STRICT-LOAD / FROZEN / FP32")
print("Student init              : S1 SEED-42 BASELINE BEST")
print("Student parameters        : 8,990,352")
print("Optimizer groups          : 3 / 3")
print("Physical/accum/effective  : 1 / 64 / 64")
print("Steps / epoch / total     :", STEPS_PER_EPOCH, "/", TOTAL_STEPS)
print("Student AMP / GradScaler  : ENABLED / 1024")
print("Optimizer step            : NOT PERFORMED")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 3 / 7 code cells")


XLKD-03 CELL 3 — MODEL AND OPTIMIZER ENGINE
Status                    : PASSED
Teacher                   : STRICT-LOAD / FROZEN / FP32
Student init              : S1 SEED-42 BASELINE BEST
Student parameters        : 8,990,352
Optimizer groups          : 3 / 3
Physical/accum/effective  : 1 / 64 / 64
Steps / epoch / total     : 1771 / 35420
Student AMP / GradScaler  : ENABLED / 1024
Optimizer step            : NOT PERFORMED
Validation / Test         : NOT READ / SEALED
Progress                  : 3 / 7 code cells


## Cell 4 — Kế thừa Validation engine đã kiểm chứng


In [4]:
STARTED = time.perf_counter()

baseline_validation_files = sorted(
    S1_PATH.parent.rglob("epoch_*_validation.json")
)
assert len(baseline_validation_files) == int(S1["completed_epochs"])
baseline_best = read_json(
    S1_PATH.parent / "validation" / "epochs"
    / f'epoch_{int(S1["best_epoch"]):02d}_validation.json'
)
assert int(baseline_best["epoch"]) == int(S1["best_epoch"])
for metric in ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"]:
    assert math.isfinite(float(baseline_best["metrics"][metric]))
assert math.isfinite(float(
    baseline_best["teacher_forced"]["mean_token_cross_entropy"]
))
assert baseline_best["data_policy"]["test_set_status"] == "sealed"
assert baseline_best["data_policy"]["test_set_used"] is False

VALIDATION_POLICY_PATH = CONFIG / "xlkd03_validation_selection_policy.json"
write_json(VALIDATION_POLICY_PATH, {
    "schema_version": "xlkd03_validation_selection_policy_v1",
    "created_at_utc": now(),
    "status": "locked_by_inheritance",
    "basis": (
        "S1 architecture and decoder are unchanged from the verified "
        "Phase-2 baseline validation/Beam-3 engine."
    ),
    "validation_caption_count": 25010,
    "validation_image_count": 5000,
    "generation": {
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_steps": 31,
    },
    "metrics": ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"],
    "selection": {
        "primary": "CIDEr-D",
        "tie_break": ["BLEU-4", "CE", "earlier epoch"],
    },
    "readiness_phase_validation_opened": False,
    "full_run_validation_authorized_only_after_xlkd03_complete": True,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "xlkd03_cell04_validation_inheritance.json", {
    "schema_version": "xlkd03_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "source": "S1 seed-42 baseline verified validation engine",
    "source_validation_epochs": len(baseline_validation_files),
    "source_best_epoch": int(S1["best_epoch"]),
    "source_best_metrics": S1["best_validation_metrics"],
    "student_inference_architecture_changed": False,
    "validation_policy_locked": True,
    "validation_opened_in_xlkd03": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 4 — VALIDATION ENGINE INHERITANCE")
print("=" * 78)
print("Status                    : PASSED")
print("Verified source epochs    :", len(baseline_validation_files))
print("Student inference change  : False")
print("Validation policy         : LOCKED")
print("Primary / tie-break       : CIDEr-D / BLEU-4 / CE / epoch")
print("Validation opened here    : False")
print("Full-run Validation       : CONDITIONAL ON XLKD-03 COMPLETE")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 7 code cells")


XLKD-03 CELL 4 — VALIDATION ENGINE INHERITANCE
Status                    : PASSED
Verified source epochs    : 20
Student inference change  : False
Validation policy         : LOCKED
Primary / tie-break       : CIDEr-D / BLEU-4 / CE / epoch
Validation opened here    : False
Full-run Validation       : CONDITIONAL ON XLKD-03 COMPLETE
Test Set status           : SEALED
Progress                  : 4 / 7 code cells


## Cell 5 — Discarded pilot và atomic exact resume


In [5]:
STARTED = time.perf_counter()

def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)

def nested_exact(left, right):
    if isinstance(left, torch.Tensor):
        return (
            isinstance(right, torch.Tensor)
            and left.dtype == right.dtype
            and left.shape == right.shape
            and torch.equal(left.cpu(), right.cpu())
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and left.keys() == right.keys()
            and all(nested_exact(left[key], right[key]) for key in left)
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(nested_exact(a, b) for a, b in zip(left, right))
        )
    return left == right

def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)

def save_checkpoint(path, optimizer, scheduler, scaler, progress):
    atomic_save(path, {
        "schema_version": "xlkd_dense_mapped_s1_checkpoint_v1",
        "experiment_id": "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGIT",
        "student_encoder_state": clone_cpu(STUDENT_ENCODER.state_dict()),
        "student_decoder_state": clone_cpu(STUDENT_DECODER.state_dict()),
        "optimizer_state": clone_cpu(optimizer.state_dict()),
        "scheduler_state": clone_cpu(scheduler.state_dict()),
        "scaler_state": clone_cpu(scaler.state_dict()),
        "progress": dict(progress),
        "cpu_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
        "student_initial_checkpoint_sha256": sha256(BASELINE_CHECKPOINT),
        "mapping_sha256": sha256(MAPPING_PATH),
        "loss_contract_sha256": sha256(LOSS_CONTRACT_PATH),
        "temperature": TEMPERATURE,
        "alpha_human_ce": ALPHA_HUMAN,
        "alpha_kd": ALPHA_KD,
        "test_set_status": "sealed",
        "test_set_used": False,
    })

reset_student_baseline()
optimizer, scheduler, scaler = build_optimizer_state()
pilot_iterator = iter(build_train_loader(epoch=0, cursor=0))
pilot_rows = []
pilot_started = time.perf_counter()
cursor = 0
overflow_total = 0
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
for step in range(1, 3):
    microbatches = [next(pilot_iterator) for _ in range(ACCUMULATION)]
    outcome = optimizer_step(
        microbatches, optimizer, scheduler, scaler, retry_limit=10
    )
    cursor += len(microbatches)
    overflow_total += outcome["overflow_retries"]
    row = {
        "optimizer_step": step,
        "cursor": cursor,
        **outcome,
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_adapter": optimizer.param_groups[1]["lr"],
        "lr_decoder": optimizer.param_groups[2]["lr"],
    }
    pilot_rows.append(row)
    print(
        f"step={step:02d} | cursor={cursor} | "
        f"loss={outcome['combined_loss']:.6f} | "
        f"CE={outcome['human_ce']:.6f} | "
        f"KL*T2={outcome['mapped_kl_t2']:.6f} | "
        f"mass_min={outcome['mapped_mass_minimum']:.6f}"
    )
pilot_seconds = time.perf_counter() - pilot_started
assert cursor == 128
assert all(math.isfinite(row["combined_loss"]) for row in pilot_rows)
assert all(row["mapped_mass_mean"] >= 0.98 for row in pilot_rows)
assert all(row["mapped_mass_minimum"] >= 0.95 for row in pilot_rows)
assert not any(parameter.grad is not None for parameter in TEACHER.parameters())
peak_gpu_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
assert peak_gpu_mib <= 14336

TEMP = CHECKPOINT / "xlkd03_discarded_pilot.pt"
progress = {"epoch": 0, "cursor": cursor, "global_optimizer_step": 2}
save_checkpoint(TEMP, optimizer, scheduler, scaler, progress)
expected_encoder = clone_cpu(STUDENT_ENCODER.state_dict())
expected_decoder = clone_cpu(STUDENT_DECODER.state_dict())
expected_optimizer = clone_cpu(optimizer.state_dict())
expected_scheduler = clone_cpu(scheduler.state_dict())
expected_scaler = clone_cpu(scaler.state_dict())
expected_cpu_rng = torch.get_rng_state().clone()
expected_cuda_rng = [
    state.clone() for state in torch.cuda.get_rng_state_all()
]

reset_student_baseline()
loaded_optimizer, loaded_scheduler, loaded_scaler = build_optimizer_state()
checkpoint = torch.load(TEMP, map_location="cpu", weights_only=False)
assert checkpoint["schema_version"] == "xlkd_dense_mapped_s1_checkpoint_v1"
STUDENT_ENCODER.load_state_dict(
    checkpoint["student_encoder_state"], strict=True
)
STUDENT_DECODER.load_state_dict(
    checkpoint["student_decoder_state"], strict=True
)
loaded_optimizer.load_state_dict(checkpoint["optimizer_state"])
loaded_scheduler.load_state_dict(checkpoint["scheduler_state"])
loaded_scaler.load_state_dict(checkpoint["scaler_state"])
torch.set_rng_state(checkpoint["cpu_rng"])
torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
assert nested_exact(expected_encoder, STUDENT_ENCODER.state_dict())
assert nested_exact(expected_decoder, STUDENT_DECODER.state_dict())
assert nested_exact(expected_optimizer, loaded_optimizer.state_dict())
assert nested_exact(expected_scheduler, loaded_scheduler.state_dict())
assert nested_exact(expected_scaler, loaded_scaler.state_dict())
assert torch.equal(expected_cpu_rng, torch.get_rng_state())
assert all(
    torch.equal(left, right)
    for left, right in zip(expected_cuda_rng, torch.cuda.get_rng_state_all())
)
TEMP.unlink()
assert not TEMP.exists()
reset_student_baseline()

PILOT_CSV = LOG / "xlkd03_discarded_pilot.csv"
with PILOT_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(pilot_rows[0]))
    writer.writeheader()
    writer.writerows(pilot_rows)

seconds_per_step = pilot_seconds / 2
estimated_hours_per_epoch = seconds_per_step * STEPS_PER_EPOCH / 3600
write_json(AUDIT / "xlkd03_cell05_discarded_pilot.json", {
    "schema_version": "xlkd03_cell05_v1",
    "created_at_utc": now(),
    "status": "passed",
    "optimizer_steps": 2,
    "microbatches": 128,
    "images": cursor,
    "overflow_retries": overflow_total,
    "seconds_per_optimizer_step": seconds_per_step,
    "estimated_hours_per_epoch": estimated_hours_per_epoch,
    "peak_gpu_memory_mib": peak_gpu_mib,
    "mapped_mass_gate": True,
    "teacher_no_gradient": True,
    "atomic_checkpoint": True,
    "resume_load_exact": True,
    "temporary_checkpoint_deleted": True,
    "student_baseline_restore_exact": True,
    "pilot_weights_saved": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 5 — DISCARDED PRODUCTION PILOT")
print("=" * 78)
print("Status                    : PASSED")
print("Optimizer steps / images  : 2 / 128")
print("Overflow retries          :", overflow_total)
print("Mapped mass gate          : PASSED")
print("Teacher no-gradient       : PASSED")
print("Peak GPU memory           :", round(peak_gpu_mib, 2), "MiB")
print("Atomic resume exact       : PASSED")
print("Temporary checkpoint      : DELETED")
print("Baseline restore          : EXACT")
print("Estimated hours / epoch   :", round(estimated_hours_per_epoch, 3))
print("Pilot weights saved       : False")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 5 / 7 code cells")


step=01 | cursor=64 | loss=10.687754 | CE=2.026857 | KL*T2=19.348650 | mass_min=0.992990
step=02 | cursor=128 | loss=10.315933 | CE=1.938378 | KL*T2=18.693488 | mass_min=0.992341
XLKD-03 CELL 5 — DISCARDED PRODUCTION PILOT
Status                    : PASSED
Optimizer steps / images  : 2 / 128
Overflow retries          : 0
Mapped mass gate          : PASSED
Teacher no-gradient       : PASSED
Peak GPU memory           : 1189.31 MiB
Atomic resume exact       : PASSED
Temporary checkpoint      : DELETED
Baseline restore          : EXACT
Estimated hours / epoch   : 10.894
Pilot weights saved       : False
Validation / Test         : NOT READ / SEALED
Progress                  : 5 / 7 code cells


## Cell 6 — Khóa production contract và lộ trình cuối


In [6]:
STARTED = time.perf_counter()

PRODUCTION_CONTRACT_PATH = (
    CONFIG / "xlkd03_dense_mapped_production_contract.json"
)
production_contract = {
    "schema_version": "xlkd03_dense_mapped_production_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "experiment_id": "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGIT",
    "scientific_role": "adaptive controlled ablation after sparse top-k failure",
    "teacher": {
        "name": "ExpansionNet v2",
        "checkpoint_sha256": sha256(CHECKPOINT_PATH),
        "mode": "eval",
        "frozen": True,
        "input_resolution": [384, 384],
        "logit_dtype": "float32",
    },
    "student": {
        "name": "S1 MobileNetV3-Large plus shared Transformer decoder",
        "initialization": "S1 seed-42 baseline best checkpoint",
        "checkpoint_sha256": sha256(BASELINE_CHECKPOINT),
        "input_resolution": [224, 224],
    },
    "fairness_disclosure": {
        "regime": "post-baseline KD fine-tuning",
        "baseline_pretraining_epochs": int(S1["completed_epochs"]),
        "additional_kd_epochs_maximum": 20,
        "claim_boundary": (
            "This branch measures the incremental benefit of mapped "
            "Response KD after baseline training. It is not an equal-total-"
            "training-budget comparison against a from-scratch baseline."
        ),
        "report_additional_gpu_hours": True,
    },
    "mapping": {
        "teacher_vocabulary": 10000,
        "mapped_teacher_dimensions": 9959,
        "student_vocabulary": 10080,
        "mapping_sha256": sha256(MAPPING_PATH),
        "unmapped_policy": "discard 41 dimensions and renormalize",
        "offline_dense_corpus": False,
        "sparse_top_k": False,
    },
    "loss": {
        "formula": (
            "0.5*CE_human + "
            "0.5*T^2*KL(q_teacher_mapped_renormalized || p_student)"
        ),
        "temperature": 4.0,
        "alpha_human_ce": 0.5,
        "alpha_kd": 0.5,
        "ce_mask": "human target labels",
        "kd_mask": "aligned pseudo-caption target labels",
        "teacher_conditioning": "native pseudo-caption IDs",
        "student_kd_conditioning": "aligned Phase-1 pseudo-caption IDs",
    },
    "training": {
        "images_per_epoch": 113287,
        "physical_batch": 1,
        "gradient_accumulation": 64,
        "effective_batch": 64,
        "maximum_epochs": 20,
        "steps_per_epoch": 1771,
        "optimizer": "AdamW",
        "scheduler": "5 percent warmup plus cosine",
        "student_mixed_precision": "FP16 autocast plus GradScaler",
        "teacher_precision": "FP32",
        "gradient_clip": 1.0,
        "overflow_retry_limit": 10,
        "atomic_last_every_optimizer_steps": 250,
        "resume_boundary": "optimizer step",
        "session_budget_hours": 10.5,
    },
    "validation": {
        "authorized_for_full_run": True,
        "caption_count": 25010,
        "image_count": 5000,
        "generation": "Beam-3 length penalty 0.7",
        "metrics": ["BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"],
        "selection_primary": "CIDEr-D",
        "tie_break": ["BLEU-4", "CE", "earlier epoch"],
    },
    "early_stopping": {
        "minimum_epoch": 5,
        "patience": 4,
        "minimum_delta": 0.001,
    },
    "comparison": {
        "primary": "S1 seed-42 CE-only baseline",
        "secondary": "ExpansionNet v2 to S1 Sequence-KD seed-42",
        "teacher_metric_is_not_kd_improvement": True,
        "equal_total_training_budget": False,
        "required_reporting": [
            "initial baseline checkpoint",
            "additional KD epochs",
            "additional optimizer steps",
            "additional GPU-hours",
        ],
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PRODUCTION_CONTRACT_PATH, production_contract)

FINAL_ROADMAP_PATH = CONFIG / "final_kd_roadmap.json"
write_json(FINAL_ROADMAP_PATH, {
    "schema_version": "klcn_final_kd_roadmap_v1",
    "created_at_utc": now(),
    "status": "locked",
    "governance": {
        "gate_before_next_phase": True,
        "seed42_before_confirmation_seeds": True,
        "retain_negative_results": True,
        "no_posthoc_threshold_relaxation": True,
        "validation_only_for_selection": True,
        "test_open_once_after_final_method_lock": True,
    },
    "stages": [
        "XLKD-03 readiness",
        "XLKD-04 response-KD seed42",
        "XLKD-05 response-KD seed123/2026",
        "XLKD-06 response-KD three-seed aggregate",
        "MLKD-00/01 feature tap, adapter and fixed-batch preflight",
        "MLKD-02/03/04 feature-KD seed42, confirmation and aggregate",
        "MLKD-05/06 full multi-level compatibility and readiness",
        "MLKD-07/08/09 full multi-level three seeds and ablation aggregate",
        "EVAL final Validation lock and one-time held-out Test",
        "MOBILE export, parity, RAM/latency/model-size benchmark",
        "REPORT thesis tables, uncertainty, negative results and limitations",
    ],
    "kd_methods": {
        "sequence_kd": ["T1->S1 control", "T2->S2"],
        "logit_response_kd": [
            "T3->S3 existing contract",
            "T1->S1 dense cross-vocabulary mapped response KD",
        ],
        "feature_kd": ["T1->S1 spatial pooling plus learned projection"],
        "full_multi_level_kd": [
            "T1->S1 hard CE plus feature KD plus mapped response KD"
        ],
    },
    "explicitly_not_used_without_new_protocol": [
        "direct raw-logit KD for T1->S1",
        "sparse top-k mapped KD",
        "Attention KD",
        "Relation KD",
        "joint multi-teacher training",
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "xlkd03_cell06_contract_roadmap.json", {
    "schema_version": "xlkd03_cell06_v1",
    "created_at_utc": now(),
    "status": "passed",
    "production_contract_sha256": sha256(PRODUCTION_CONTRACT_PATH),
    "roadmap_sha256": sha256(FINAL_ROADMAP_PATH),
    "production_contract_locked": True,
    "roadmap_locked": True,
    "validation_selection_authorized_for_full_run": True,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 6 — PRODUCTION CONTRACT AND FINAL ROADMAP")
print("=" * 78)
print("Status                    : PASSED")
print("Production contract       : LOCKED")
print("Final roadmap             : LOCKED")
print("Maximum epochs            : 20")
print("Session/resume            : 10.5 h / OPTIMIZER-STEP EXACT")
print("Validation selection      : AUTHORIZED FOR XLKD-04")
print("Held-out Test             : SEALED")
print("Full training             : NOT STARTED")
print("Progress                  : 6 / 7 code cells")


XLKD-03 CELL 6 — PRODUCTION CONTRACT AND FINAL ROADMAP
Status                    : PASSED
Production contract       : LOCKED
Final roadmap             : LOCKED
Maximum epochs            : 20
Session/resume            : 10.5 h / OPTIMIZER-STEP EXACT
Validation selection      : AUTHORIZED FOR XLKD-04
Held-out Test             : SEALED
Full training             : NOT STARTED
Progress                  : 6 / 7 code cells


## Cell 7 — Package XLKD-03 và authorize XLKD-04 seed 42


In [7]:
STARTED = time.perf_counter()

AUTH_NEXT = OUT / "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_AUTHORIZED.json"
write_json(AUTH_NEXT, {
    "schema_version": "xlkd04_dense_mapped_seed42_full_run_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "experiment_id": "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGIT",
    "seed": 42,
    "required_next": "XLKD-04 seed-42 full run with exact resume",
    "production_contract": {
        "relative_path": PRODUCTION_CONTRACT_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(PRODUCTION_CONTRACT_PATH),
    },
    "roadmap": {
        "relative_path": FINAL_ROADMAP_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(FINAL_ROADMAP_PATH),
    },
    "validation_selection_authorized": True,
    "full_training_authorized": True,
    "confirmation_seeds_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})

tracked = sorted(
    path for path in OUT.rglob("*")
    if path.is_file()
    and not path.is_relative_to(MANIFEST)
    and not path.name.endswith(".tmp")
)
manifest_rows = [{
    "relative_path": path.relative_to(OUT).as_posix(),
    "size_bytes": path.stat().st_size,
    "sha256": sha256(path),
} for path in tracked]
MANIFEST_CSV = MANIFEST / "xlkd03_manifest.csv"
MANIFEST_JSON = MANIFEST / "xlkd03_manifest.json"
SHA256SUMS = MANIFEST / "SHA256SUMS"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(MANIFEST_JSON, {
    "schema_version": "xlkd03_manifest_v1",
    "created_at_utc": now(),
    "artifact_count": len(manifest_rows),
    "artifacts": manifest_rows,
})
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

COMPLETE = OUT / "XLKD03_DENSE_MAPPED_TRAINING_READINESS_COMPLETE.json"
write_json(COMPLETE, {
    "schema_version": "xlkd03_dense_mapped_training_readiness_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "experiment_id": "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGIT",
    "seed": 42,
    "production_data_pipeline": "passed",
    "model_optimizer_engine": "passed",
    "validation_policy_inheritance": "passed",
    "discarded_pilot": "passed",
    "atomic_resume": "passed",
    "production_contract": "locked",
    "final_roadmap": "locked",
    "xlkd04_seed42_authorized": True,
    "authorization_marker": {
        "relative_path": AUTH_NEXT.relative_to(OUT).as_posix(),
        "sha256": sha256(AUTH_NEXT),
    },
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(OUT).as_posix(),
            "sha256": sha256(MANIFEST_CSV),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": MANIFEST_JSON.relative_to(OUT).as_posix(),
            "sha256": sha256(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(OUT).as_posix(),
            "sha256": sha256(SHA256SUMS),
        },
    },
    "pilot_weights_saved": False,
    "validation_read": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("XLKD-03 FINAL STATUS")
print("=" * 78)
print("Status                    : COMPLETE")
print("Production data/model     : PASSED / PASSED")
print("Validation policy         : INHERITED AND LOCKED")
print("Discarded pilot           : 2 / 2 PASSED")
print("Atomic resume             : PASSED")
print("Production contract       : LOCKED")
print("Final roadmap             : LOCKED")
print("Manifest rows             :", len(manifest_rows))
print("XLKD-04 seed42            : AUTHORIZED")
print("Pilot weights saved       : False")
print("Validation opened here    : False")
print("Test Set status           : SEALED")
print("Progress                  : 7 / 7 code cells")


XLKD-03 FINAL STATUS
Status                    : COMPLETE
Production data/model     : PASSED / PASSED
Validation policy         : INHERITED AND LOCKED
Discarded pilot           : 2 / 2 PASSED
Atomic resume             : PASSED
Production contract       : LOCKED
Final roadmap             : LOCKED
Manifest rows             : 12
XLKD-04 seed42            : AUTHORIZED
Pilot weights saved       : False
Validation opened here    : False
Test Set status           : SEALED
Progress                  : 7 / 7 code cells
